# BeltWatch: train on Kaggle

Builds the dataset and trains **one** neural model per run, then evaluates it on val. See `docs/training.md`.

**Before the first run**, in the notebook's right-hand panel (Session options):
- **Accelerator:** GPU T4 x2. Avoid the P100: recent CUDA builds of PyTorch may not support it.
- **Internet:** on. Kaggle needs a phone-verified account for this.

**To run:** set `MODEL` below, then **Save Version → Save & Run All (Commit)**. The run continues in the background without an open browser tab and saves `/kaggle/working/beltwatch-runs` as the version's output. Each run takes about 5 to 6 hours of the 12-hour limit: roughly 1 hour of data preparation, then about 4.5 hours of training.

**If a run stops early:** open the notebook, choose **Add Input → Your Work → this notebook**, select the stopped version's output, and commit again. The notebook copies that output back and training resumes from `latest.pt`.

In [ ]:
import os
import shutil

MODEL = "unet"  # "unet" or "segformer": one full training run per Kaggle session

WORK = "/kaggle/working/beltwatch-runs"  # saved as the version's output (20 GB limit)
MODELS = {
    "unet": ("configs/unet.yaml", "unet-resnet18-ce"),
    "segformer": ("configs/segformer_b0.yaml", "segformer-b0-ce"),
}
BASE_CONFIG, RUN_DIR_NAME = MODELS[MODEL]


def scratch_root() -> str:
    # The dataset (about 16 GB with extraction) stays off /kaggle/working, which is published
    # as output. Use whichever local disk has the most space.
    candidates = [p for p in ("/kaggle/temp", "/tmp", "/root") if os.path.isdir(p)]
    return max(candidates, key=lambda p: shutil.disk_usage(p).free)


REPO = os.path.join(scratch_root(), "beltwatch")
os.makedirs(WORK, exist_ok=True)
print("model:", MODEL, "| outputs:", WORK, "| repo:", REPO)
!df -h /kaggle/working {os.path.dirname(REPO)}

## Restore a previous run's output (only when resuming)

In [ ]:
def find_previous_output() -> str | None:
    # Kaggle mounts an attached notebook output somewhere under /kaggle/input.
    for root, dirs, _ in os.walk("/kaggle/input"):
        if root.count(os.sep) > 5:
            dirs.clear()
        elif os.path.basename(root) == "beltwatch-runs":
            return root
    return None


previous = find_previous_output()
if previous and not os.listdir(WORK):
    shutil.copytree(previous, WORK, dirs_exist_ok=True)
    print("restored", previous)
else:
    print("no previous output attached" if not previous else "outputs already present")

## Set up

In [ ]:
if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/professor3333/beltwatch.git {REPO}
%cd {REPO}
!git log -1 --oneline
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv sync --extra cu126
!uv run python -c "import torch; ok = torch.cuda.is_available(); print('CUDA:', ok, torch.cuda.get_device_name(0) if ok else ''); assert ok, 'no GPU: set Accelerator to GPU T4 x2'"

## Build the dataset
Downloads and verifies the pinned 7.5 GB archive (resumable), then validates, audits duplicates and assigns splits. The dataset-version files are copied into the output so they can be committed through a PR.

In [ ]:
!uv run dvc repro
!mkdir -p {WORK}/dataset-version
!cp -r data/manifests data/splits dvc.lock {WORK}/dataset-version/
!grep -m1 split_manifest_id data/splits/zerowaste-f-splits.json

## Train and evaluate on val

In [ ]:
import yaml

RUN_DIR = f"{WORK}/models/{RUN_DIR_NAME}"
with open(BASE_CONFIG) as fh:
    config = yaml.safe_load(fh)
config["output_dir"] = RUN_DIR
config["tracking"]["mlflow_uri"] = f"sqlite:///{WORK}/mlflow.db"
config["runtime"]["num_workers"] = min(4, os.cpu_count() or 1)
with open("/kaggle/working/train-config.yaml", "w") as fh:
    yaml.safe_dump(config, fh)

RESUME = f"--resume {RUN_DIR}/latest.pt" if os.path.exists(f"{RUN_DIR}/latest.pt") else ""
print("resume:", RESUME or "no, starting fresh")
!uv run python scripts/train.py --config /kaggle/working/train-config.yaml {RESUME}
!uv run python scripts/evaluate.py --model {MODEL} --model-dir {RUN_DIR} --split val --out-dir {WORK}/reports

## Afterwards
Download the version's output (`beltwatch-runs/`). The files to bring back are `mlflow.db`, `reports/`, `dataset-version/`, and `models/<run>/best.pt` with `run.json`. `latest.pt` is only needed to resume.